<a href="https://colab.research.google.com/github/Abdulla4akash/traffictwin/blob/claude/complete-v0.7/gpu/colab/bcap_synthetic_smoke.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TrafficTwin B-CAP synthetic GPU smoke

This is the **first Colab engineering experiment**, not a scientific experiment. It trains two small synthetic contextual-bandit policies to prove the JAX/GPU, multi-seed, 17-D/19-D observation, manifest, checkpoint, and return-artifact path.

Every output is labelled `synthetic_engineering_diagnostic_only`, `scientific_evidence=false`, and `actor_admission_eligible=false`.

## Hard boundary

Until Randy's written Colab permission is recorded, this notebook must not receive `vec_env`, `tos-data`, checkpoints, real traces, university GitLab content, bus artifacts, `.demo/`, `data/vec-fresh/`, or quarantine bytes. Raw BODS snapshots never enter Colab.

The notebook clones only the TrafficTwin GitHub repository and generates every training/evaluation context in memory. Do not add Drive mounts or upload cells.

## 1. Select a GPU runtime, then install the public pinned JAX stack

In Colab choose **Runtime → Change runtime type → GPU** before running this cell. The install uses only public PyPI packages; it does not retrieve project data or producer assets.

In [ ]:
%pip install -q --upgrade "jax[cuda12]==0.4.30" "numpy==1.26.4"


In [ ]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import jax

devices = jax.devices()
print("JAX", jax.__version__)
print("devices", devices)
if not any(device.platform in {"gpu", "cuda", "rocm"} for device in devices):
    raise RuntimeError("No JAX GPU detected. Select a Colab GPU runtime and rerun from cell 1.")

## 2. Clone only TrafficTwin

The resolved commit is recorded in `execution_manifest.json`. The cell refuses to replace an existing directory; use a fresh runtime rather than silently changing source beneath a run.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/Abdulla4akash/traffictwin.git"
REPOSITORY_REF = "claude/complete-v0.7"
REPOSITORY_ROOT = Path("/content/traffictwin-bcap-smoke")

if REPOSITORY_ROOT.exists():
    raise RuntimeError(f"Refusing to replace existing source directory: {REPOSITORY_ROOT}")
subprocess.run(  # noqa: S603 - fixed public TrafficTwin clone command
    [
        "/usr/bin/git", "clone", "--depth", "1", "--branch",
        REPOSITORY_REF, REPOSITORY_URL, str(REPOSITORY_ROOT),
    ],
    check=True,
)
resolved_commit = subprocess.run(  # noqa: S603 - fixed read-only git command
    ["/usr/bin/git", "rev-parse", "HEAD"],
    cwd=REPOSITORY_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print("TrafficTwin commit", resolved_commit)
sys.path.insert(0, str(REPOSITORY_ROOT))

## 3. Run the matched five-seed synthetic diagnostic

The control actor receives 17 synthetic features with capacity hidden. The treatment actor receives the same 17 features plus normalized capacity and RSU headroom. Both use the same toy reward, training budget, capacity grid, and fixed evaluation contexts. This demonstrates wiring only; it does not estimate a TrafficTwin effect.

In [ ]:
from gpu.colab.bcap_synthetic_smoke import SmokeConfig, run_experiment

OUTPUT_DIR = Path("/content/bcap-synthetic-smoke-output")
config = SmokeConfig(
    model_seeds=(100, 101, 102, 103, 104),
    evaluation_seeds=(9000, 9001, 9002),
    updates=250,
    batch_size=2048,
    evaluation_batch_size=4096,
    require_gpu=True,
)
result = run_experiment(config, OUTPUT_DIR)
print("output", result["output_dir"])
print("design fingerprint", result["design_fingerprint"])
print("scientific evidence", result["summary"]["scientific_evidence"])

## 4. Inspect diagnostics without promoting them to results

In [ ]:
import json

summary = json.loads((OUTPUT_DIR / "summary.json").read_text())
rows = []
for variant, variant_summary in summary["variants"].items():
    row = {"variant": variant, "model_count": variant_summary["model_count"]}
    for metric, values in variant_summary["metrics"].items():
        row[f"{metric}_mean"] = values["mean"]
        row[f"{metric}_population_sd"] = values["population_sd"]
    rows.append(row)
print(json.dumps(rows, indent=2))
print(summary["interpretation"])

In [ ]:
design = json.loads((OUTPUT_DIR / "design_manifest.json").read_text())
execution = json.loads((OUTPUT_DIR / "execution_manifest.json").read_text())
inventory = json.loads((OUTPUT_DIR / "output_inventory.json").read_text())
print("status:", design["status"])
print("actor admission eligible:", design["actor_admission_eligible"])
print("producer assets used:", design["producer_assets_used"])
print("bus assets used:", design["bus_assets_used"])
print("resolved commit:", execution["repository_commit"])
print("hashed output files:", len(inventory["files"]))

## 5. Bundle the synthetic return artifacts

Downloading this ZIP is safe because the driver has no external input path. It remains a diagnostic bundle and must never enter the actor allowlist or evidence registry.

In [ ]:
import shutil

archive = shutil.make_archive(
    "/content/bcap-synthetic-smoke-return",
    "zip",
    root_dir=OUTPUT_DIR,
)
print(archive)
# Optional manual download:
# from google.colab import files
# files.download(archive)

## Stop boundary

The first Colab smoke ends here. Do not adapt this notebook to real producer assets. The next real B-CAP step is a separately reviewed observation/state implementation and signed predeclaration after permission—not changing a path in this notebook.